# Customer Churn Analysis & Retention Strategy

This notebook follows the six-phase plan in the project README. Run cells in order using the project environment. Current code covers loading and initial profiling only; later phases are planning sections.

**Business objective:** Identify customer segments worth investigating and develop evidence-based retention recommendations. Initial profiles are not final business findings.


## Phase 1: Business framing and data audit

**Question:** Is the data suitable for the planned analysis?

Load the raw file without modifying it. Review identifiers, missing and blank values, numeric conversion issues, and outcome labels. Record source verification and audit decisions in `outputs/reports/analysis_log.md`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Support execution from the repository root or any directory inside it.
working_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "datasets" / "TelecomChurnData.csv").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run this notebook from inside the CustomerChurnPrediction repository.")

raw_data_path = project_root / "datasets" / "TelecomChurnData.csv"
raw_df = pd.read_csv(raw_data_path)
display(raw_df.head())


In [ ]:
overview = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "unique_customer_ids": raw_df["customerID"].nunique(),
        "duplicate_rows": int(raw_df.duplicated().sum()),
        "rows_with_repeated_customer_id": int(
            raw_df["customerID"].duplicated(keep=False).sum()
        ),
    },
    name="value",
)
display(overview.to_frame())


In [ ]:
# Profile whitespace-only values separately from pandas missing values.
blank_counts = raw_df.apply(
    lambda column: column.astype("string").str.strip().eq("").sum()
)
quality_profile = pd.DataFrame(
    {
        "dtype": raw_df.dtypes.astype(str),
        "missing_values": raw_df.isna().sum(),
        "blank_strings": blank_counts,
        "unique_non_null_values": raw_df.nunique(),
    }
)
display(quality_profile)

numeric_issues = []
for field in ["tenure", "MonthlyCharges", "TotalCharges"]:
    values = raw_df[field].astype("string").str.strip()
    converted = pd.to_numeric(values, errors="coerce")
    nonblank = values.notna() & values.ne("")
    numeric_issues.append(
        {
            "field": field,
            "nonblank_conversion_failures": int((nonblank & converted.isna()).sum()),
            "negative_values": int((converted < 0).sum()),
        }
    )
display(pd.DataFrame(numeric_issues).set_index("field"))


In [ ]:
# Preliminary frequencies only; final metrics require a validated customer grain.
churn_counts = raw_df["Churn"].value_counts(dropna=False)
churn_profile = pd.DataFrame(
    {
        "row_count": churn_counts,
        "share_of_rows_pct": churn_counts.div(len(raw_df)).mul(100),
    }
)
display(churn_profile)

invalid_churn_labels = int((~raw_df["Churn"].isin(["Yes", "No"])).sum())
print(f"Rows with missing or unexpected churn labels: {invalid_churn_labels}")


## Phase 2: Data cleaning and analytical preparation

**Planned:** Investigate the audit results, document treatments, validate customer uniqueness, and define the churn indicator and analytical bands. Reconcile counts before and after preparation.

Keep `raw_df` unchanged and create a separate analytical table. Save generated tables in `outputs/data/` only after the cleaning rules are agreed and documented. Fit learned model preprocessing on training data in Phase 5.


## Phase 3: Exploratory analysis and SQL validation

**Planned:** Answer the README's business questions using contract, tenure, charges, billing, and service comparisons. Show group sizes, churn volumes, and churn rates together.

Add reproducible queries in `sql/`, reconcile their outputs with pandas, and save reviewed charts in `outputs/charts/`. Record evidence and limitations in the analysis log.


## Phase 4: Business interpretation and retention planning

**Planned:** Prioritise segments by evidence, scale, and operational feasibility. Distinguish observed associations from explanations and proposed actions.

Use the recommendation template in the analysis log to define the target segment, action, owner, success measure, and pilot design. Campaign execution requires additional operational data.


## Phase 5: Baseline predictive support

**Planned:** Verify feature timing, split before fitting preprocessing, and compare a dummy classifier, logistic regression, and a simple segmentation rule.

Choose thresholds using training-side validation. Evaluate held-out precision, recall, and lift at a documented contact capacity. Record limitations and any generated artifacts in `outputs/models/`; add model dependencies when implementing this phase.


## Phase 6: Dashboard, reporting, and portfolio delivery

**Planned:** Build the executive overview, segment analysis, and retention priorities pages described in `dashboard/README.md`.

Reconcile dashboard metrics with Python and SQL, write the executive summary, and update the project README with validated findings and artifact links.
